# Pesos óptimos del ridge filter (mínimos cuadrados acotados)

Calcula los pesos (anchos de escalón) del ridge que dan la dosis más plana, a partir de la base $B_t(x)$ simulada con láminas uniformes de PLA (casos `slab_tNN`, NN = t en décimas de mm: 00, 02, ..., 22).

$$D(x) = \sum_i w_i B_{t_i}(x), \qquad w_i \ge w_{min}, \qquad \sum_i w_i = 1, \qquad t_i = 0, 0.2, \ldots, (n-1)\cdot 0.2\ \text{mm}$$

Se minimiza $\|A w - 1\|^2$ sobre la ventana $[D - T_{max}, D]$ y luego se normaliza $w$; el nivel de dosis es libre.

## 1. Imports

In [ ]:
import glob, os, re
import numpy as np
from types import SimpleNamespace
from scipy.optimize import lsq_linear

## 2. Parámetros
Reemplazan a los argumentos de `argparse` del script original.

In [ ]:
a = SimpleNamespace(
    glob="casos/slab_t*/out/bragg_pouch_z.out",  # archivos de la base (--glob)
    front=60.28,        # posición del frente del gel [cm] (--front)
    D=2.5,              # profundidad fija de la pileta [mm] (--D)
    Tmax=2.25,          # ancho de la ventana de ajuste [mm] (--Tmax)
    Ts="1.0,1.25,1.5,2.0,2.25",  # ventanas T para las que se reporta el CV (--Ts)
    period=0.63,        # período del ridge [cm] (--period)
    res=0.0025,         # resolución de la impresora [cm] (--res)
    block=0,            # bloque del archivo de salida a leer (--block)
    nmax=12,            # máximo de espesores a usar (--nmax)
    minw=0.0025,        # ancho completo mínimo por escalón [cm]; 25 um = resolución de la impresora (--minw)
)

## 3. Definición de funciones

In [ ]:
def load(path, block, front):
    """Lee un archivo .out y devuelve (x en mm desde el frente, dosis) del bloque pedido."""
    blocks = [[]]
    for l in open(path):
        t = l.split()
        if len(t) == 4:
            try:
                v = [float(x) for x in t]
            except ValueError:
                continue
            if blocks[-1] and v[0] < blocks[-1][-1][0]:
                blocks.append([])
            blocks[-1].append(v)
    R = np.array(blocks[block])
    return (0.5*(R[:, 0] + R[:, 1]) - front)*10.0, R[:, 2]


def analytic(N, p=1.827):
    """Pesos analíticos para un ridge de N escalones."""
    aa = 1 - 1/p
    ww = np.array([((i+0.5)**aa - max(i-0.5, 0)**aa)/aa for i in range(N)])
    return ww/ww.sum()


def rounded(wt, res, period):
    """Redondea los anchos a múltiplos de la resolución de impresión y compensa en el escalón 0."""
    full = np.round(wt*period/res)*res
    full[0] += period - full.sum()
    return full/period

## 4. Carga de la base de láminas

In [ ]:
files = sorted(glob.glob(a.glob))
basis = {}
for f in files:
    m = re.search(r"slab_t(\d\d)", f)
    if not m:
        continue
    t = int(m.group(1))/10.0
    basis[round(t, 1)] = load(f, a.block, a.front)

ts = sorted(basis)[:a.nmax]
if len(ts) < 3:
    raise RuntimeError("faltan casos slab_tNN: encontrados %s" % ts)
print("Espesores cargados (mm):", ts)

## 5. Ajuste por mínimos cuadrados acotados

In [ ]:
xg = np.arange(0.0, a.D + 0.3, 0.02)
A = np.array([np.interp(xg, *basis[t]) for t in ts]).T   # columnas = B_t en la grilla común
win = (xg >= a.D - a.Tmax) & (xg <= a.D)
wmin = a.minw/a.period   # un escalón con peso 0 no se puede construir: cota inferior = ancho mínimo imprimible

sol = lsq_linear(A[win], np.ones(win.sum()), bounds=(np.full(len(ts), wmin), np.inf))
w = sol.x
lvl = 1.0/w.sum()
w = w/w.sum()

## 6. Candidatos a comparar

In [ ]:
def cv(wt, T):
    """Coeficiente de variación (%) de la dosis en la ventana [D-T, D]."""
    s = (xg >= a.D - T) & (xg <= a.D)
    d = A[s] @ wt
    return d.std()/d.mean()*100

Ts = [float(v) for v in a.Ts.split(",")]

wr = rounded(w, a.res, a.period)
cands = [("minimos cuadrados (continuo)", w),
         ("min. cuadrados redond. %.0f um" % (a.res*1e4), wr)]

for N in (9, 10, 11, 12):
    if N <= len(ts):
        cands.append(("analitico N%d" % N, np.pad(analytic(N), (0, len(ts)-N))))

orig = np.array([0.18, 0.09, 0.07, 0.06, 0.05, 0.05, 0.05, 0.04, 0.04])
orig = orig/orig.sum()
if len(ts) >= 9:
    cands.append(("ridge original 9", np.pad(orig, (0, len(ts)-9))))

## 7. Resultados

In [ ]:
print("Espesores de PLA disponibles (mm):", ts)
print("Ventana de ajuste: x en [%.2f, %.2f] mm desde el frente del gel; pileta fija D = %.2f mm\n" % (a.D-a.Tmax, a.D, a.D))
print("Pesos por espesor (fraccion de area):")
print("  t(mm)   " + "  ".join("%5.1f" % t for t in ts))
for name, wt in cands:
    print("  %-30s" % name + "  ".join("%5.3f" % v for v in wt))

In [ ]:
print("CV (%) de la dosis en la ventana [D-T, D]:")
print("  %-30s" % "T (mm)" + "  ".join("%6.2f" % T for T in Ts))
for name, wt in cands:
    print("  %-30s" % name + "  ".join("%6.2f" % cv(wt, T) for T in Ts))

## 8. Anchos completos para `gen_ridge.py --widths`

In [ ]:
full = (wr*a.period)[::-1]   # del más grueso (índice n-1) al de espesor 0
keep = [x for x in full]
while keep and keep[0] <= 0:   # descartar escalones gruesos con peso 0
    keep.pop(0)

print("anchos completos (cm) para gen_ridge.py --widths, escalon 1 (grueso) -> espesor 0 (%d escalones, tmax %.1f mm):" % (len(keep), 0.2*(len(keep)-1)))
print(",".join("%.4f" % x for x in keep))
print("nivel de dosis relativo del ridge optimizado = %.3f (unidades del tally por fuente)" % lvl)